In [29]:
from ingest import load_faq_data
documents = load_faq_data()

In [30]:
documents[10]

{'id': '316180784f',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: How many hours per week am I expected to spend on this course?',
 'answer': 'It depends on your background and previous experience with modules. It is expected to require about 5 - 15 hours per week.\n\nYou can also calculate it yourself using [this data](https://github.com/DataTalksClub/zoomcamp-analytics/tree/main/data/de-zoomcamp-2023) and then update this answer.'}

In [31]:
documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

len(documents_llm)

113

In [32]:
documents = documents_llm

In [33]:
doc = documents[0]
print(doc["id"])
print(doc["question"])
print(doc["answer"])

74eb249bbf
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.


In [34]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

In [35]:
data_gen_instructions = """
You emulate a student who's taking our course.
Formulate 5 questions this student might ask based on a FAQ record. The record
should contain the answer to the questions, and the questions should be complete and not too short.
If possible, use as fewer words as possible from the record.

The output should resemble how people ask questions
on the internet. Not too formal, not too short, not too long.
""".strip()

In [36]:
from dotenv import load_dotenv
from google import genai
from google.genai.types import HttpOptions

load_dotenv()

PROJECT_ID = "llm-zoomcamp-500505"
LOCATION = "us-central1"

client = genai.Client(
    vertexai=True,
    project=PROJECT_ID,
    location=LOCATION,
    http_options=HttpOptions(api_version="v1"),
)

In [37]:
import json
user_prompt = json.dumps(doc)

In [10]:
user_prompt

'{"id": "74eb249bbf", "course": "llm-zoomcamp", "section": "General Course-Related Questions", "question": "I just discovered the course. Can I still join?", "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\\u2019re still accepting submissions."}'

In [11]:
messages = [
    {"role": "developer", "content": data_gen_instructions},
    {"role": "user", "content": user_prompt}
]

In [38]:
from google.genai import types

response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=messages[1]["content"],
    config=types.GenerateContentConfig(
        system_instruction=messages[0]["content"],
        response_mime_type="application/json",
        response_schema=Questions,
    ),
)

# الحصول على النتيجة
result = getattr(response, "parsed", None)

if result is None:
    result = Questions.model_validate_json(response.text)

In [39]:
print(result.questions)

['I just found out about the course, can I still sign up?', 'If I join late, is it still possible to get a certificate?', 'What do I need to do to receive a course certificate?', 'Is there a time limit for submitting the project to get certified?', 'Can I still earn a certificate if I submit my project after joining late?']


In [40]:
result = getattr(response, "parsed", None)

if result is None:
    result = Questions.model_validate_json(response.text)

print(result.questions)

['I just found out about the course, can I still sign up?', 'If I join late, is it still possible to get a certificate?', 'What do I need to do to receive a course certificate?', 'Is there a time limit for submitting the project to get certified?', 'Can I still earn a certificate if I submit my project after joining late?']


In [41]:
doc

{'id': '74eb249bbf',
 'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'I just discovered the course. Can I still join?',
 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'}

In [42]:
len(documents_llm)

113

In [43]:
from evaluation_utils import llm_structured

In [44]:
result, usage = llm_structured(
    client,
    data_gen_instructions,
    user_prompt,
    Questions
)

print(result.questions)

ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'Resource exhausted. Please try again later. Please refer to https://cloud.google.com/vertex-ai/generative-ai/docs/error-code-429 for more details.', 'status': 'RESOURCE_EXHAUSTED'}}

In [46]:
usage

NameError: name 'usage' is not defined

In [20]:
from evaluation_utils import calc_price

In [21]:
calc_price(usage)

NameError: name 'usage' is not defined

In [ ]:
print(usage)

cache_tokens_details=None cached_content_token_count=None candidates_token_count=84 candidates_tokens_details=[ModalityTokenCount(
  modality=<MediaModality.TEXT: 'TEXT'>,
  token_count=84
)] prompt_token_count=179 prompt_tokens_details=[ModalityTokenCount(
  modality=<MediaModality.TEXT: 'TEXT'>,
  token_count=179
)] thoughts_token_count=350 tool_use_prompt_token_count=None tool_use_prompt_tokens_details=None total_token_count=613 traffic_type=<TrafficType.ON_DEMAND: 'ON_DEMAND'>


In [22]:
records = []

for q in result.questions:
    records.append({
        "question": q,
        "document": doc["id"]
    })

records

[{'question': 'Is it still possible to sign up for this course?',
  'document': '74eb249bbf'},
 {'question': 'What do I need to do to get a certificate for completing the course?',
  'document': '74eb249bbf'},
 {'question': 'If I want a certificate, is there a specific deadline for submitting my project?',
  'document': '74eb249bbf'},
 {'question': 'Do I have to submit a project to receive a certificate?',
  'document': '74eb249bbf'},
 {'question': 'Can I still earn a certificate if I join the course after it has already started?',
  'document': '74eb249bbf'}]

In [24]:
import evaluation_utils

print(evaluation_utils.__file__)

/workspaces/llm-zoomcamp-2026-code/04-evaluation/code/evaluation_utils.py


In [25]:
import inspect
import evaluation_utils

print(inspect.getsource(evaluation_utils.calc_price))

def calc_price(usage):
    input_tokens = usage.prompt_token_count or 0
    output_tokens = usage.candidates_token_count or 0

    input_cost = (input_tokens / 1_000_000) * INPUT_PRICE_PER_MILLION
    output_cost = (output_tokens / 1_000_000) * OUTPUT_PRICE_PER_MILLION

    return {
        "input_tokens": input_tokens,
        "output_tokens": output_tokens,
        "total_tokens": usage.total_token_count,
        "input_cost": input_cost,
        "output_cost": output_cost,
        "total_cost": input_cost + output_cost,
    }



In [26]:
import importlib
import evaluation_utils

importlib.reload(evaluation_utils)

<module 'evaluation_utils' from '/workspaces/llm-zoomcamp-2026-code/04-evaluation/code/evaluation_utils.py'>

In [27]:
cost = evaluation_utils.calc_price(usage)
print(cost)

NameError: name 'usage' is not defined

In [28]:
# cost = calc_price(usage)

print(cost)

NameError: name 'cost' is not defined

In [ ]:
records = []

for q in result.questions:
    records.append({
        "question": q,
        "document": doc["id"]
    })

records

[{'question': 'I just found out about the course, is it too late to enroll?',
  'document': '74eb249bbf'},
 {'question': 'If I join now, can I still earn a certificate of completion?',
  'document': '74eb249bbf'},
 {'question': "What's required for me to get a certificate if I'm starting the course late?",
  'document': '74eb249bbf'},
 {'question': 'Is there a specific submission window for the project if I want to receive a certificate?',
  'document': '74eb249bbf'},
 {'question': "What's the latest I can turn in my project to be eligible for a certificate?",
  'document': '74eb249bbf'}]

In [ ]:
import pandas as pd

In [ ]:
pd.DataFrame(records)

,question,document
0,Is it still possible to join the course?,74eb249bbf
1,"If I start late, can I still get a certificate?",74eb249bbf
2,What do I need to do to receive a certificate ...,74eb249bbf
3,Does the project submission window affect cert...,74eb249bbf
4,When is the final deadline to submit my projec...,74eb249bbf


In [ ]:
from evaluation_utils import llm_structured_retry

In [ ]:
def generate_ground_truth(doc):
    user_prompt = json.dumps(doc)

    out, usage = llm_structured_retry(
        client,
        data_gen_instructions,
        user_prompt,
        Questions
    )

    results = []

    for q in out.questions:
        results.append({
            "question": q,
            "document": doc["id"]
        })

    return results, usage

In [ ]:
generate_ground_truth(doc)

([{'question': 'Is it still possible to join the course now that it has started?',
   'document': '74eb249bbf'},
  {'question': 'What do I need to do to earn a certificate for completing the course?',
   'document': '74eb249bbf'},
  {'question': 'Is there a specific deadline for submitting my project if I want a certificate?',
   'document': '74eb249bbf'},
  {'question': 'If I enrolled late, can I still get a certificate?',
   'document': '74eb249bbf'},
  {'question': "What's the key requirement for receiving a completion certificate in this course?",
   'document': '74eb249bbf'}],
 GenerateContentResponseUsageMetadata(
   candidates_token_count=98,
   candidates_tokens_details=[
     ModalityTokenCount(
       modality=<MediaModality.TEXT: 'TEXT'>,
       token_count=98
     ),
   ],
   prompt_token_count=179,
   prompt_tokens_details=[
     ModalityTokenCount(
       modality=<MediaModality.TEXT: 'TEXT'>,
       token_count=179
     ),
   ],
   thoughts_token_count=621,
   total_toke

In [ ]:
from tqdm.auto import tqdm

ground_truth = []
usages = []

for doc in tqdm(documents[:1]):
    records, usage = generate_ground_truth(doc)
    ground_truth.extend(records)
    usages.append(usage)

  0%|          | 0/1 [00:00<?, ?it/s]

ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'Resource exhausted. Please try again later. Please refer to https://cloud.google.com/vertex-ai/generative-ai/docs/error-code-429 for more details.', 'status': 'RESOURCE_EXHAUSTED'}}

In [ ]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

In [ ]:
from concurrent.futures import ThreadPoolExecutor

with ThreadPoolExecutor(max_workers=1) as pool:
    results = map_progress(pool, documents, generate_ground_truth)

  0%|          | 0/103 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
ground_truth = []
usages = []

for records, usage in results:
    ground_truth.extend(records)
    usages.append(usage)

len(ground_truth)

NameError: name 'results' is not defined

In [ ]:
ground_truth[10]

{'question': 'How do I join the Office Hours or live workshop if I don’t have the Zoom link?',
 'document': '489dd1c9d9'}

In [ ]:
from evaluation_utils import calc_price

total_cost = 0.0

for usage in usages:
    cost = calc_price(usage)
    total_cost = total_cost + cost["total_cost"]

total_cost

0.05718675000000001

In [ ]:
from evaluation_utils import calc_total_price

calc_total_price(usages)

0.05718675000000001

In [ ]:
df_ground_truth = pd.DataFrame(ground_truth)

In [ ]:
df_ground_truth.to_csv("data/ground_truth.csv", index=False)

In [ ]:
len(df_ground_truth)

395